# 📈 Alpaca Trading Bot — phone-ready (Colab)

Momentum + volume scanner with a **confirm-before-you-buy** step. Defaults to **paper trading** (fake money).

**How to use (just tap the ▶ button on each cell, top to bottom):**
1. **Cell 1** – install (tap ▶, wait ~30s).
2. **Cell 2** – paste your Alpaca **paper** API key + secret into the boxes, then tap ▶.
3. **Cell 3** – tap ▶ once to load the engine.
4. **`balance`** cell – tap ▶ to check you're connected.
5. **`scan`** cell – tap ▶ to find one candidate trade.
6. **`buy`** cell – set CONFIRM to `yes` and tap ▶ to place the scanned trade.

Get free paper keys at **app.alpaca.markets** → switch to *Paper Trading* → *Generate API Keys*.

> 🔒 Keys live only in this notebook session. Don't share the notebook after pasting them in.

### 1) Install (tap ▶)

In [ ]:
!pip install -q "alpaca-py>=0.30.0"
print("✅ Installed. Now run cell 2.")

### 2) Your settings — paste keys, then tap ▶
Everything you can tune is here. Keep **LIVE = false** while testing.

In [ ]:
#@title Settings { display-mode: "form" }
ALPACA_API_KEY    = ""  #@param {type:"string"}
ALPACA_API_SECRET = ""  #@param {type:"string"}
LIVE              = "false"  #@param ["false", "true"]
POSITION_SIZE_PCT = 0.05  #@param {type:"number"}
MAX_ORDER_DOLLARS = 1000  #@param {type:"number"}
LOOKBACK_DAYS     = 20  #@param {type:"integer"}

LIVE = (LIVE == "true")

# Universe: ~30 liquid large/mid-cap US stocks. Edit freely.
UNIVERSE = [
    "AAPL","MSFT","GOOGL","AMZN","META","NVDA","TSLA","AVGO",
    "AMD","INTC","QCOM","MU",
    "JPM","BAC","GS","V","MA",
    "JNJ","UNH","PFE","LLY",
    "WMT","COST","HD","MCD","NKE","SBUX",
    "CAT","BA","XOM","CVX",
]

assert ALPACA_API_KEY and ALPACA_API_SECRET, "Paste your API key AND secret above, then run this cell again."
if LIVE:
    print("!"*60)
    print("!!  ⚠  LIVE TRADING IS ON — ORDERS USE REAL MONEY  ⚠")
    print("!!  Set LIVE back to 'false' to use the safe paper sandbox.")
    print("!"*60)
else:
    print("[PAPER MODE] Safe sandbox — fake money. ✅")
print("Settings loaded. Now run cell 3.")

### 3) Load the engine (tap ▶ once)
This defines the strategy, sizing, and Alpaca connection. **The strategy lives in `score_symbol()`** — scroll in and edit it to tune.

In [ ]:
from datetime import datetime, timedelta, timezone
from dataclasses import dataclass
from typing import List, Optional

from alpaca.data.enums import DataFeed
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.trading.client import TradingClient
from alpaca.trading.enums import OrderSide, TimeInForce
from alpaca.trading.requests import MarketOrderRequest

# Connect. paper=not LIVE is the key safety switch.
trading = TradingClient(ALPACA_API_KEY, ALPACA_API_SECRET, paper=not LIVE)
data = StockHistoricalDataClient(ALPACA_API_KEY, ALPACA_API_SECRET)

@dataclass
class Bar:
    close: float
    volume: float

@dataclass
class SymbolScore:
    symbol: str; score: float; last_price: float; sma: float
    momentum_strength: float; volume_ratio: float; reason: str

# ======================= THE STRATEGY (edit me) =======================
# Candidate must be (1) above its N-day moving average (uptrend) AND
# (2) trading on above-average volume today. Ranked by strength * volume.
def score_symbol(symbol, bars):
    if len(bars) < 2:
        return None
    closes = [b.close for b in bars]; vols = [b.volume for b in bars]
    last = closes[-1]
    sma = sum(closes)/len(closes)
    avg_vol = sum(vols)/len(vols)
    today_vol = vols[-1]
    if sma <= 0 or avg_vol <= 0:
        return None
    momentum = (last - sma)/sma          # % above the average
    vol_ratio = today_vol/avg_vol        # how heavy today is
    if not (last > sma and today_vol > avg_vol):   # both filters must pass
        return None
    score = momentum * vol_ratio
    reason = (f"Price ${last:.2f} is {momentum*100:.1f}% above its "
              f"{len(bars)}-day SMA (${sma:.2f}); volume {vol_ratio:.2f}x average.")
    return SymbolScore(symbol, score, last, sma, momentum, vol_ratio, reason)
# ======================================================================

def fetch_bars(symbols):
    cal_days = int(LOOKBACK_DAYS*1.6)+7
    end = datetime.now(timezone.utc); start = end - timedelta(days=cal_days)
    req = StockBarsRequest(symbol_or_symbols=symbols, timeframe=TimeFrame.Day,
                           start=start, end=end, feed=DataFeed.IEX)
    raw = getattr(data.get_stock_bars(req), "data", {}) or {}
    out = {}
    for sym, sdk_bars in raw.items():
        if not sdk_bars: continue
        recent = sdk_bars[-LOOKBACK_DAYS:]
        out[sym] = [Bar(float(b.close), float(b.volume)) for b in recent]
    return out

def size_position(price, buying_power):
    if price <= 0 or buying_power <= 0:
        return 0, 0.0, "No buying power / bad price."
    budget = min(POSITION_SIZE_PCT*buying_power, MAX_ORDER_DOLLARS, buying_power)
    qty = int(budget // price)
    if qty < 1:
        return 0, 0.0, f"Budget ${budget:,.2f} < one share at ${price:,.2f}."
    exp = (f"{POSITION_SIZE_PCT*100:.1f}% of ${buying_power:,.2f} = ${POSITION_SIZE_PCT*buying_power:,.2f} "
           f"target, capped at ${MAX_ORDER_DOLLARS:,.2f} -> budget ${budget:,.2f}; "
           f"{qty} share(s) (~${qty*price:,.2f}).")
    return qty, qty*price, exp

LAST_CANDIDATE = None   # set by scan(), read by the buy cell
print("✅ Engine loaded. Try the 'balance' cell, then 'scan'.")

### 💰 balance — tap ▶

In [ ]:
try:
    a = trading.get_account()
    print(f"Status         : {a.status}")
    print(f"Buying power   : ${float(a.buying_power):,.2f}")
    print(f"Cash           : ${float(a.cash):,.2f}")
    print(f"Portfolio value: ${float(a.portfolio_value):,.2f}")
except Exception as e:
    print("Error:", e)
    print("Check your keys in cell 2 (and that LIVE matches the key type).")

### 📊 positions — tap ▶

In [ ]:
try:
    ps = trading.get_all_positions()
    if not ps:
        print("No open positions.")
    else:
        for p in ps:
            print(f"{p.symbol:<6} qty {float(p.qty):<8.2f} "
                  f"avg ${float(p.avg_entry_price):<8.2f} now ${float(p.current_price):<8.2f} "
                  f"P/L ${float(p.unrealized_pl):,.2f}")
except Exception as e:
    print("Error:", e)

### 🔍 scan — tap ▶
Finds the single best candidate. It does **not** buy — it just shows you the trade. To place it, use the **buy** cell below.

In [ ]:
LAST_CANDIDATE = None
try:
    if not trading.get_clock().is_open:
        print("Note: market is CLOSED. Any order will queue until the next open.\n")
except Exception as e:
    print("(couldn't check market hours:", e, ")\n")

print(f"Scanning {len(UNIVERSE)} tickers...")
try:
    bars = fetch_bars(UNIVERSE)
except Exception as e:
    bars = {}; print("Market data error:", e)

scored = [s for s in (score_symbol(sym, b) for sym, b in bars.items()) if s]
scored.sort(key=lambda s: s.score, reverse=True)

if not scored:
    print("No candidates passed the strategy filters right now. Nothing to trade.")
else:
    held = {p.symbol for p in trading.get_all_positions()}
    c = scored[0]
    if c.symbol in held:
        print(f"Top pick {c.symbol} is already held — skipping to avoid stacking.")
    else:
        bp = float(trading.get_account().buying_power)
        qty, cost, how = size_position(c.last_price, bp)
        print("="*56)
        print(f"CANDIDATE: {c.symbol}")
        print(f"Price   : ${c.last_price:,.2f}")
        print(f"Signal  : {c.reason}")
        print(f"Score   : {c.score:.4f}")
        print("-"*56)
        if qty < 1:
            print("Proposed order: NONE —", how)
        else:
            print(f"Proposed order: BUY {qty} share(s) of {c.symbol}  (~${cost:,.2f})")
            print(f"Sizing  : {how}")
            LAST_CANDIDATE = (c.symbol, qty, c.last_price)
            print("="*56)
            print("👉 To place this, set CONFIRM = yes in the BUY cell below and run it.")

### ✅ buy — set CONFIRM to `yes`, then tap ▶
Places the trade from the most recent **scan**. Nothing happens while CONFIRM is `no`.

In [ ]:
#@title Place the scanned order { display-mode: "form" }
CONFIRM = "no"  #@param ["no", "yes"]

if LAST_CANDIDATE is None:
    print("Run the 'scan' cell first to get a candidate.")
elif CONFIRM != "yes":
    sym, qty, price = LAST_CANDIDATE
    print(f"Ready to BUY {qty} {sym} (~${qty*price:,.2f}). "
          f"Set CONFIRM to 'yes' above and run again to place it.")
else:
    sym, qty, price = LAST_CANDIDATE
    try:
        order = trading.submit_order(order_data=MarketOrderRequest(
            symbol=sym, qty=qty, side=OrderSide.BUY, time_in_force=TimeInForce.DAY))
        st = order.status.value if hasattr(order.status, "value") else order.status
        print("✅ Order submitted!")
        print("   Order ID:", order.id)
        print("   Symbol  :", order.symbol)
        print("   Qty     :", order.qty)
        print("   Status  :", st)
        LAST_CANDIDATE = None  # prevent accidental double-buy
    except Exception as e:
        print("Order failed:", e)